# CCM-v2 / SmolLM2-360M — **real fast evaluation of the ALREADY TRAINED checkpoint**

**No re-training. No teacher-response acceptance gate.** Load the ZIP from your previous CCM-v2 experiment (including `best_ccm_consensus_layer0.pt`), then evaluate **gold labeled ARC-Easy** questions, WikiText-2 gold next-token loss/accuracy, transparent behavioral chat prompts, and actual paired CUDA prefill latency.

Original SmolLM2 is the comparison baseline—not the authority deciding whether an answer is correct. The evaluation is reproducible and saves per-case details and a ZIP. **Only layer 0 is cellular**; every other decoder layer remains native. The cellular consensus mathematically resembles softmax attention over a bounded tape, and no native cache is supported.

Repository: [TinyCeNN-LM research branch](https://github.com/vtavakkoli/TinyCeNN-LM/tree/research/competitive-cenn-feasibility). Keep the experiment out of production until independent held-out quality/latency succeeds.

In [ ]:
# @title 1. Dependencies and checkout (GPU required)
%pip -q install "transformers==5.17.0" "datasets>=3,<5" "huggingface_hub>=0.36,<2" "safetensors" "accelerate" "pandas" "pytest"
import os,sys,subprocess,json,zipfile,io,time,math,hashlib
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Use Runtime > Change runtime type > GPU'
REPO=Path('/content/TinyCeNN-LM')
BRANCH='research/competitive-cenn-feasibility'
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','--quiet','--depth','1','--branch',BRANCH,
        'https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','--quiet','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','--quiet',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/'+BRANCH],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'--no-deps'],check=True)
if str(REPO) not in sys.path: sys.path.insert(0,str(REPO))
if str(REPO/'src') not in sys.path: sys.path.insert(0,str(REPO/'src'))
from scripts.benchmark_ccm_consensus_smol360 import (ConsensusAttentionAdapter,
    attention_slot,compare_full_model,load_blocks,paired_latency)
from scripts.real_fast_eval import arc_easy_eval,real_chat_eval,CHAT_CASES
from tinycenn_lm.competitive_consensus_v2 import ConsensusConfig,CompetitiveConsensusCeNN
from transformers import AutoModelForCausalLM,AutoTokenizer
import pandas as pd
from IPython.display import display
DEVICE=torch.device('cuda')
DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print('GPU',torch.cuda.get_device_name(0),'Torch',torch.__version__)


In [ ]:
# @title 2. Upload previous experiment ZIP (no retraining)
RESULTS_ZIP = '' # @param {type:'string'}
if RESULTS_ZIP and Path(RESULTS_ZIP).is_file():
    archive=Path(RESULTS_ZIP)
else:
    from google.colab import files
    print('Upload CCM_v2_360M_Consensus_Chat_Results.zip from the earlier Colab run')
    uploaded=files.upload()
    assert len(uploaded)==1,'Upload exactly one CCM results ZIP'
    archive=Path('/content')/next(iter(uploaded))
assert archive.is_file() and zipfile.is_zipfile(archive)
with zipfile.ZipFile(archive) as z:
    assert 'best_ccm_consensus_layer0.pt' in z.namelist(), 'Missing trained checkpoint'
    raw=z.read('best_ccm_consensus_layer0.pt')
    assert len(raw)<200_000_000,'Unusually large checkpoint'
    ckpt=torch.load(io.BytesIO(raw),map_location='cpu',weights_only=True)
    prior_report=json.loads(z.read('final_report.json')) if 'final_report.json' in z.namelist() else None
assert all(k in ckpt for k in ('config','core','revision','step'))
print('Trained checkpoint step',ckpt['step'],'revision',ckpt['revision'])
if prior_report:
    assert prior_report['revision']==ckpt['revision'],'Checkpoint/report revision mismatch'
    print('Earlier test gold NLL native/CCM',prior_report['test']['teacher_nll'],prior_report['test']['student_nll'])


In [ ]:
# @title 3. Load pinned original backbone and install ONLY layer-0 cellular checkpoint
MODEL_ID='HuggingFaceTB/SmolLM2-360M-Instruct'
from huggingface_hub import model_info
revision=ckpt['revision']
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=revision)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=revision,
    torch_dtype=DTYPE,attn_implementation='sdpa').to(DEVICE).eval()
model.requires_grad_(False)
assert model.config.model_type=='llama'
native=model.model.layers[0].self_attn
config=ConsensusConfig(**ckpt['config'])
assert config.hidden_size==model.config.hidden_size
core=CompetitiveConsensusCeNN(config).to(DEVICE).eval()
core.load_state_dict(ckpt['core'],strict=True)
adapter=ConsensusAttentionAdapter(core).eval()
assert model.model.layers[0].self_attn is native
print('Pure replacement module (no original full attention call):',type(core).__name__)
print('Native layer params',sum(p.numel() for p in native.parameters()),
      'CCM layer params',sum(p.numel() for p in core.parameters()))


In [ ]:
# @title 4. Real ground-truth ARC-Easy multiple-choice evaluation
from datasets import load_dataset
ARC_QUESTIONS = 16 # @param {type:'integer'}
EVAL_SEED = 2031 # @param {type:'integer'}
assert 1 <= ARC_QUESTIONS <= 200
arc_data=load_dataset('allenai/ai2_arc','ARC-Easy',split='validation')
with attention_slot(model,native):
    arc_native=arc_easy_eval(model,tokenizer,dataset=arc_data,limit=ARC_QUESTIONS,seed=EVAL_SEED)
with attention_slot(model,adapter):
    arc_ccm=arc_easy_eval(model,tokenizer,dataset=arc_data,limit=ARC_QUESTIONS,seed=EVAL_SEED)
assert arc_native['evaluated']==arc_ccm['evaluated']>0
assert [r['id'] for r in arc_native['rows']]==[r['id'] for r in arc_ccm['rows']]
print('GOLD-labeled ARC-Easy accuracy native',arc_native['accuracy'],'CCM',arc_ccm['accuracy'])
display(pd.DataFrame([{'id':a['id'],'gold':a['gold'],
 'native_choice':a['pred'],'CCM_choice':b['pred'],
 'native_correct':a['correct'],'CCM_correct':b['correct']}
 for a,b in zip(arc_native['rows'],arc_ccm['rows'])]))


In [ ]:
# @title 5. Gold next-token NLL, gold accuracy, and actual CUDA prefill latency
# WikiText-2 TEST was previously inspected when selecting this checkpoint.
# It remains true labeled data but is NOT an untouched independent test anymore.
WIKITEXT_BLOCKS=16 # @param {type:'integer'}
CONTEXT_LENGTH=128 # @param {type:'integer'}
assert 1<=WIKITEXT_BLOCKS<=256 and CONTEXT_LENGTH>=16
ids=load_blocks(tokenizer,'test',WIKITEXT_BLOCKS,CONTEXT_LENGTH)
metrics=compare_full_model(model,native,adapter,ids,batch_size=1,device=DEVICE)
latency=paired_latency(model,native,adapter,ids[:1],DEVICE)
assert model.model.layers[0].self_attn is native
print('REAL gold NLL: native',metrics['teacher_nll'],'CCM',metrics['student_nll'])
print('REAL gold token accuracy: native',metrics['teacher_gold_accuracy'],
      'CCM',metrics['student_gold_accuracy'])
print('NLL delta',metrics['delta_nll'], 'full-model prefill GPU speedup',latency['speedup'])
print('Teacher top-1 agreement (not a criterion)',metrics['teacher_agreement'])


In [ ]:
# @title 6. Rule-scored chat evaluation — answers judged by task, not teacher match
CHAT_MAX_TOKENS=48 # @param {type:'integer'}
assert 1<=CHAT_MAX_TOKENS<=160
with attention_slot(model,native):
    chat_native=real_chat_eval(model,tokenizer,cases=CHAT_CASES,max_new_tokens=CHAT_MAX_TOKENS)
with attention_slot(model,adapter):
    chat_ccm=real_chat_eval(model,tokenizer,cases=CHAT_CASES,max_new_tokens=CHAT_MAX_TOKENS)
assert model.model.layers[0].self_attn is native
print('Task-rule pass rate native',chat_native['accuracy'],'CCM',chat_ccm['accuracy'])
display(pd.DataFrame([{'task':x['id'],'native_correct':x['pass'],
 'CCM_correct':y['pass'],'native_response':x['response'],'CCM_response':y['response']}
 for x,y in zip(chat_native['rows'],chat_ccm['rows'])]))


In [ ]:
# @title 7. Optional custom chat question — preview side by side
CUSTOM_QUESTION = '' # @param {type:'string'}
if CUSTOM_QUESTION.strip():
    from scripts.real_fast_eval import greedy_no_cache
    with attention_slot(model,native):
        a=greedy_no_cache(model,tokenizer,CUSTOM_QUESTION,max_new_tokens=CHAT_MAX_TOKENS)
    with attention_slot(model,adapter):
        b=greedy_no_cache(model,tokenizer,CUSTOM_QUESTION,max_new_tokens=CHAT_MAX_TOKENS)
    print('ORIGINAL:',a['response'])
    print('CCM:',b['response'])
else:
    print('Set CUSTOM_QUESTION for a personal chat comparison')


In [ ]:
# @title 8. Strict gold-data quality/speed audit + ZIP export
from scripts.real_fast_eval import quality_gate
OUTPUT=Path('/content/CCM_v2_RealFast_Only_Results')
OUTPUT.mkdir(exist_ok=True)
quality=quality_gate(None,metrics,latency,max_delta_nll=0.05,
                     max_gold_accuracy_drop=0.02,min_speedup=1.0)
# Tiny ARC/chat sample sizes are descriptive only, not deployment gate evidence.
report={'model':MODEL_ID,'revision':revision,'checkpoint_step':int(ckpt['step']),
  'mode':'trained-checkpoint-only; no training',
  'real_world_gold_wikitext':metrics,'note':'WikiText-2 test already inspected by previous CCM run',
  'real_arc_easy':{'native':arc_native,'CCM':arc_ccm},
  'real_chat_rule_scored':{'native':chat_native,'CCM':chat_ccm},
  'paired_cuda_prefill':latency,'gold_ground_truth_gate':quality,
  'gold_and_speed_pass':bool(quality['quality_pass'] and quality['speed_pass']),
  'teacher_agreement_used_for_decision':False,
  'limits':'Only first attention submodule replaced; other Llama blocks native; no cache; not a novel softmax-free algorithm; ARC is small sample'}
(OUTPUT/'real_fast_report.json').write_text(json.dumps(report,indent=2))
(OUTPUT/'arc_comparisons.json').write_text(json.dumps({'native':arc_native,'CCM':arc_ccm},indent=2))
(OUTPUT/'chat_behavior.json').write_text(json.dumps({'native':chat_native,'CCM':chat_ccm},indent=2))
(OUTPUT/'checkpoint_sha256.txt').write_text(hashlib.sha256(raw).hexdigest()+'\n')
print('GOLD+GPU pass:',report['gold_and_speed_pass'],'(teacher agreement NOT used)')
print('No model is published by this experiment.')
final_zip=OUTPUT.parent/(OUTPUT.name+'.zip')
with zipfile.ZipFile(final_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUTPUT.rglob('*')):
        if p.is_file():z.write(p,str(p.relative_to(OUTPUT)))
print('Results ZIP',final_zip)
try:
    from google.colab import files
    files.download(str(final_zip))
except ImportError:
    print('Retrieve from Colab Files:',final_zip)


### Interpretation

- **Gold token NLL/accuracy:** quantitative language-model quality. Native is a control, not the authority on how the student must phrase a reply.
- **ARC-Easy:** independent labeled multiple-choice diagnostic; limited sample has high sampling uncertainty and is *not* an official leaderboard result.
- **Chat cases:** explicit, reproducible correctness rubrics for facts, arithmetic, exact formatting and short in-prompt recall. Do not equate a five-question pass rate with general chatbot performance.
- **Latency:** matched GPU full-model prefill with `use_cache=False`. Do not use this to claim streaming generation acceleration.
- **Architecture:** the CCM layer only replaces `model.model.layers[0].self_attn`. It performs cellular consensus equivalent to bounded-key softmax-weighted aggregation; no claim of an original attention-free operator is established.
